# 02 — Persiapan Dataset Admiralty 29 Piantan

Notebook ini BELUM mengunci Skema I–VIII. Tujuannya memilih dan memvalidasi satu window 29 x 24 = 696 observasi. Rumus Skema I–VIII harus mengikuti satu referensi Admiralty yang dipakai dalam skripsi.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT=Path.cwd()
if not (PROJECT_ROOT/'src').exists(): PROJECT_ROOT=PROJECT_ROOT.parent
sys.path.insert(0,str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None
from backend.pasut.data_loader import load_month, MONTH_SHEETS
from backend.pasut.cleaning import clean_month
from backend.pasut.preprocessing import build_annual_dataset

monthly_clean={m:clean_month(load_month(m)) for m in MONTH_SHEETS}
df=build_annual_dataset(monthly_clean,year=2025)
df=df.dropna(subset=['tinggi']).sort_values('timestamp').reset_index(drop=True)
print('Total data:',len(df))

In [ ]:
# Pilihan awal untuk pengujian: 1–29 Januari 2025.
# Setelah desain penelitian disepakati, periode ini dapat diganti.
start=pd.Timestamp('2025-01-01 01:00',tz='Asia/Jakarta')
end=pd.Timestamp('2025-01-29 24:00',tz='Asia/Jakarta')
window=df[(df.timestamp>=start)&(df.timestamp<=end)].copy()

print('Periode:',window.timestamp.min(),'s.d.',window.timestamp.max())
print('Jumlah:',len(window))
print('Unique timestamp:',window.timestamp.nunique())
print('Missing tinggi:',window.tinggi.isna().sum())
display(window.head())
display(window.tail())

In [ ]:
expected=pd.date_range(start=start,end=end,freq='h')
missing=expected.difference(pd.DatetimeIndex(window.timestamp))
print('Expected:',len(expected))
print('Missing timestamp:',len(missing))
if len(missing): print(missing)

## Setelah window 696 data lolos QC

Baru lanjut ke Skema I–VIII. Jangan mencampur tabel faktor dari referensi berbeda. Untuk Skema VIII, gunakan tabel konstanta/koreksi yang benar-benar berasal dari referensi yang dipilih.